# B4 — External validation on ProteinGym: where the predictor stops working

**Question.** Does the SH3-trained folding head transfer to proteins it has never seen, and how
does that compare with zero-shot ESM-2?

**What it does.** Scores ProteinGym v1.3 substitution assays in three pre-defined blocks:
**primary** (external stability, 3 assays), **secondary** (expression/abundance, 15 assays),
**sanity** (Tsuboyama assays = training data, 10 assays; a wiring check, not a result). Assays are
excluded from the first two blocks by `DMS_id` **and** by UniProt accession if they overlap
training. Full-length proteins are scored through a 128-residue window centred on the mutation.

**Inputs:** B1 models, A2 MegaScale features, `input/datasets/DMS_*`. **Runtime:** about 15 min, **T4 GPU**.
**Outputs:** `proteingym_results.csv`, `RESULTS_B4.md`. Explanation: `docs/B4_external_validation.md`.

In [1]:
# ── setup (identical in every notebook) ─────────────────────────────────────────
ROOT   = '/content/drive/MyDrive/sh3_repro'   # the Drive folder from UPLOAD_GUIDE.md
RESUME = True                                  # reuse finished steps after a disconnect

import os, sys, json, time, importlib
from google.colab import drive; drive.mount('/content/drive')
os.system('pip -q install biopython')
sys.path.insert(0, f'{ROOT}/code')
import sh3_common as C; importlib.reload(C)
import numpy as np, pandas as pd
R = C.Run(ROOT, 'B4', 'External validation (ProteinGym)', resume=RESUME)
if C.DEV == 'cpu':
    print('WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU.')
print(C.env_info())

Mounted at /content/drive
B4 — External validation (ProteinGym)
  results -> /content/drive/MyDrive/sh3_repro/results/B4_external_validation
{'python': '3.13.15', 'numpy': '2.1.3', 'pandas': '2.2.3', 'scipy': '1.16.3', 'sklearn': '1.6.1', 'transformers': '5.16.1', 'Bio': '1.88', 'torch': '2.11.0+cu128', 'gpu': 'Tesla T4'}


In [2]:
import zipfile, torch
from Bio.Align import substitution_matrices
R.verify_inputs(['input/datasets/DMS_ProteinGym_substitutions.zip', 'input/datasets/DMS_substitutions.csv'])
FOLD, SC_M, SC_SD = C.load_fold_models(R.res('B1', 'models.pt'))
MG = pd.read_parquet(R.res('A2', 'features_megascale.parquet'), columns=['uniprot'])
B1M = json.load(open(R.res('B1', 'metrics_B1.json')))['metrics']

  verified input/datasets/DMS_ProteinGym_substitutions.zip  (3s)
  verified input/datasets/DMS_substitutions.csv  (0s)


In [3]:
if R.done('proteingym_results.csv'):
    PG = pd.read_csv(R.out('proteingym_results.csv')); print('loaded from this folder')
else:
    from transformers import AutoTokenizer, AutoModelForMaskedLM
    tok = AutoTokenizer.from_pretrained('facebook/esm2_t33_650M_UR50D')
    esm = AutoModelForMaskedLM.from_pretrained('facebook/esm2_t33_650M_UR50D').to(C.DEV).eval()
    if C.DEV == 'cuda': esm = esm.half()
    AAS = list('ACDEFGHIKLMNPQRSTVWY'); AAPOS = {a: i for i, a in enumerate(AAS)}
    AA_IDX = torch.tensor([tok.convert_tokens_to_ids(a) for a in AAS]).to(C.DEV)
    B62 = substitution_matrices.load('BLOSUM62'); KD, VOL, CHG = C.KD, C.VOL, C.CHG
    ref = pd.read_csv(f'{R.IN}/datasets/DMS_substitutions.csv')
    sel = [c for c in ref.columns if 'selection' in c.lower()][0]
    upc = [c for c in ref.columns if c.lower() in ('uniprot_id', 'uniprot')][0]
    ncol = [c for c in ref.columns if c.lower() in ('dms_total_number_mutants', 'dms_number_single_mutants')]
    TRAIN_UP = set(MG.uniprot.dropna().astype(str).str.upper())
    ts = ref.DMS_id.str.contains('Tsuboyama', case=False, na=False); up = ref[upc].astype(str).str.upper().isin(TRAIN_UP)
    stab = ref[sel].astype(str).str.lower().str.contains('stability')
    expr = ref[sel].astype(str).str.lower().str.contains('expression|abundance')
    pick = lambda t, k: t.sort_values(ncol[0], ascending=False).head(k) if ncol else t.head(k)
    WANT = pd.concat([pick(ref[stab & ~ts & ~up], 25).assign(block='primary'),
                      pick(ref[expr & ~ts & ~up], 15).assign(block='secondary'),
                      pick(ref[stab & ts], 10).assign(block='sanity')])
    Z = zipfile.ZipFile(f'{R.IN}/datasets/DMS_ProteinGym_substitutions.zip')
    names = {os.path.basename(n): n for n in Z.namelist() if n.endswith('.csv')}
    WANT = WANT[WANT.DMS_id.map(lambda i: f'{i}.csv' in names)]
    print('assays:', WANT.block.value_counts().to_dict())
    Wn = 128
    @torch.no_grad()
    def window_feats(seq, pos):
        lo = max(0, pos - 1 - Wn // 2); hi = min(len(seq), lo + Wn); lo = max(0, hi - Wn); sub = seq[lo:hi]; p = pos - 1 - lo
        enc = tok(sub, return_tensors='pt').to(C.DEV)
        rep = esm(**enc, output_hidden_states=True).hidden_states[-1][0, 1 + p].float().cpu().numpy()
        ids = enc['input_ids'][0].clone().unsqueeze(0); ids[0, 1 + p] = tok.mask_token_id
        lp = torch.log_softmax(esm(input_ids=ids, attention_mask=torch.ones_like(ids)).logits[0, 1 + p].float(), -1)
        return rep, lp[AA_IDX].cpu().numpy(), len(sub), (p + 1) / len(sub)
    def score(rep, lp, dlen, rpos, wt, mut):
        e = lp[AAPOS[mut]] - lp[AAPOS[wt]]
        sc = [e, lp[AAPOS[wt]], float(-(np.exp(lp) * lp).sum()), rpos, dlen, KD[mut] - KD[wt], VOL[mut] - VOL[wt],
              CHG.get(mut, 0) - CHG.get(wt, 0), float(B62[wt][mut]) if (wt in B62.alphabet and mut in B62.alphabet) else 0.0,
              float(mut == 'P'), float(wt == 'G'), float(mut == 'G')]
        x = np.concatenate([rep, np.array(sc, np.float32)])[None]
        return float(C.fold_scores(FOLD, x, SC_M, SC_SD, clip=20).mean()), e
    PG = []; t0 = time.time()
    for _, a in WANT.iterrows():
        d = pd.read_csv(Z.open(names[f'{a.DMS_id}.csv']))
        mc = [c for c in d.columns if c.lower() in ('mutant', 'mutants')][0]
        d = d[~d[mc].astype(str).str.contains(':')]
        if len(d) > 1500: d = d.sample(1500, random_state=0)
        ex = d[mc].astype(str).str.extract(r'^([A-Z])(\d+)([A-Z])$').dropna(); d = d.loc[ex.index]
        wt, pos, mut = ex[0].values, ex[1].astype(int).values, ex[2].values; seqs = d['mutated_sequence'].astype(str).values
        ok = np.array([0 < p <= len(s) and s[p-1] == m for s, p, m in zip(seqs, pos, mut)])
        if ok.mean() < 0.9: continue
        wt, pos, mut, seqs, y = wt[ok], pos[ok], mut[ok], seqs[ok], d.DMS_score.values[ok]
        cache = {}; pr = []; es = []
        for s_, p_, w_, m_ in zip(seqs, pos, wt, mut):
            s_ = s_[:p_-1] + w_ + s_[p_:]
            if (len(s_), p_) not in cache: cache[(len(s_), p_)] = window_feats(s_, p_)
            a_, b_ = score(*cache[(len(s_), p_)], w_, m_); pr.append(a_); es.append(b_)
        PG.append(dict(DMS_id=a.DMS_id, block=a.block, n=len(y), seq_len=len(seqs[0]),
                       rho_model=C.spearman(y, pr), rho_esm=C.spearman(y, es)))
        print(f'  {a.DMS_id[:44]:<44} {a.block:<9} model {PG[-1]["rho_model"]:+.3f} esm {PG[-1]["rho_esm"]:+.3f}  {(time.time()-t0)/60:.1f} min')
    PG = pd.DataFrame(PG); PG.to_csv(R.out('proteingym_results.csv'), index=False)
    del esm; torch.cuda.empty_cache()

config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.61GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/539 [00:00<?, ?it/s]

assays: {'secondary': 15, 'sanity': 10, 'primary': 3}
  PRKN_HUMAN_Clausen_2023                      primary   model -0.455 esm +0.406  0.9 min
  ESTA_BACSU_Nutschel_2020                     primary   model -0.340 esm +0.325  1.2 min
  AACC1_PSEAI_Dandage_2018                     primary   model -0.361 esm +0.493  1.5 min
  S22A1_HUMAN_Yee_2023_abundance               secondary model -0.469 esm +0.605  2.3 min
  PPARG_HUMAN_Majithia_2016                    secondary model -0.188 esm +0.558  3.0 min
  HXK4_HUMAN_Gersing_2023_abundance            secondary model -0.360 esm +0.289  3.7 min
  CAS9_STRP1_Spencer_2017_positive             secondary model -0.085 esm +0.035  5.2 min
  OXDA_RHOTO_Vanella_2023_expression           secondary model -0.400 esm +0.273  5.7 min
  CP2C9_HUMAN_Amorosi_2021_abundance           secondary model -0.584 esm +0.611  6.4 min
  CCR5_HUMAN_Gill_2023                         secondary model -0.269 esm +0.352  6.9 min
  PTEN_HUMAN_Matreyek_2021                    

## Results

In [4]:
PG['model_wins'] = PG.rho_model.abs() > PG.rho_esm.abs()
BL = PG.groupby('block').agg(assays=('DMS_id', 'size'), model_median_abs_rho=('rho_model', lambda x: x.abs().median()),
                             esm_median_abs_rho=('rho_esm', lambda x: x.abs().median()), model_wins=('model_wins', 'sum')).reset_index()
for _, r in BL.iterrows():
    for c in ('assays', 'model_median_abs_rho', 'esm_median_abs_rho', 'model_wins'): R.metric(f'{r.block}_{c}', r[c])
ext = PG[PG.block.isin(['primary', 'secondary'])]
ih, ie = B1M['per_domain_median_rho'], abs(B1M['esm_per_domain_median_rho'])
drop = pd.DataFrame([dict(block=b, head_drop=ih - g_.rho_model.abs().median(), esm_drop=ie - g_.rho_esm.abs().median())
                     for b, g_ in ext.groupby('block')])
drop['ratio'] = drop.head_drop / drop.esm_drop
for _, r in drop.iterrows(): R.metric(f'{r.block}_drop_ratio_head_vs_esm', r.ratio)
LEN = ext.assign(length=np.where(ext.seq_len <= 250, '≤250', '>250')).groupby('length').agg(
    assays=('DMS_id', 'size'), model=('rho_model', lambda x: x.abs().median()), esm=('rho_esm', lambda x: x.abs().median())).reset_index()
R.section('By block', '|Spearman ρ| per assay, medians. The sanity block is the training distribution: a wiring check.', BL, '{:.3f}')
R.section('Transfer drop from the internal SH3 result', f'Internal: head {ih:.3f}, ESM {ie:.3f} (per-domain medians, B1).', drop, '{:.3f}')
R.section('By sequence length (primary + secondary)', 'Small n and confounded with assay type; shown only to rule out '
          'the explanation that the 128-residue window is the problem.', LEN, '{:.3f}')
R.section('Every assay', '', PG.drop(columns='model_wins'), '{:.3f}')
R.finish([f"Primary (external stability): head {BL.set_index('block').loc['primary','model_median_abs_rho']:.3f} vs ESM "
          f"{BL.set_index('block').loc['primary','esm_median_abs_rho']:.3f}, head wins {int(BL.set_index('block').loc['primary','model_wins'])}/"
          f"{int(BL.set_index('block').loc['primary','assays'])}.",
          f"Secondary (abundance): head {BL.set_index('block').loc['secondary','model_median_abs_rho']:.3f} vs ESM "
          f"{BL.set_index('block').loc['secondary','esm_median_abs_rho']:.3f}, wins {int(BL.set_index('block').loc['secondary','model_wins'])}/"
          f"{int(BL.set_index('block').loc['secondary','assays'])}.",
          f"Sanity (training data): {BL.set_index('block').loc['sanity','model_median_abs_rho']:.3f} vs "
          f"{BL.set_index('block').loc['sanity','esm_median_abs_rho']:.3f} — the pipeline is wired correctly.",
          'Claim: a domain-scale predictor validated within its regime. Not claimed: a general stability predictor.'])


B4: REPRODUCED
                      quantity  this_run  recorded    status
                primary_assays  3.000000  3.000000 identical
  primary_model_median_abs_rho  0.361497  0.361497 identical
    primary_esm_median_abs_rho  0.405933  0.405933 identical
            primary_model_wins  2.000000  2.000000 identical
                 sanity_assays 10.000000 10.000000 identical
   sanity_model_median_abs_rho  0.939661  0.939661 identical
     sanity_esm_median_abs_rho  0.580314  0.580314 identical
             sanity_model_wins 10.000000 10.000000 identical
              secondary_assays 15.000000 15.000000 identical
secondary_model_median_abs_rho  0.399674  0.399674 identical
  secondary_esm_median_abs_rho  0.490735  0.490735 identical
          secondary_model_wins  6.000000  6.000000 identical

wrote RESULTS_B4.md, metrics_B4.json, comparison_B4.csv


,quantity,this_run,recorded,status
0,primary_assays,3.000000,3.000000,identical
1,primary_model_median_abs_rho,0.361497,0.361497,identical
2,primary_esm_median_abs_rho,0.405933,0.405933,identical
3,primary_model_wins,2.000000,2.000000,identical
4,sanity_assays,10.000000,10.000000,identical
5,sanity_model_median_abs_rho,0.939661,0.939661,identical
6,sanity_esm_median_abs_rho,0.580314,0.580314,identical
7,sanity_model_wins,10.000000,10.000000,identical
8,secondary_assays,15.000000,15.000000,identical
9,secondary_model_median_abs_rho,0.399674,0.399674,identical
